# Antioch setup for `humanoid_legs_5`

This is a **terminal runbook** for an existing clone on your local computer. Copy Bash blocks into the VS Code terminal. Antioch's CLI sends the project to a managed GPU session; Isaac Sim and Isaac Lab run there, not in this local `uv` environment. The robot is still an inspection asset, not a validated walking model.


## 1. Select the deployment branch in your existing clone

Run this **inside the `simulation` Git repository**, at the level where `README.md` is visible. Check for local edits first; do not discard them to switch branches.

```bash
cd /path/to/your/simulation
git status -sb
git fetch origin
git switch --track origin/antioch/humanoid_legs_5
git pull --ff-only
```

If the local branch already exists, use `git switch antioch/humanoid_legs_5` instead. If Git refuses because you have local changes, stop and preserve those changes before switching. `warehouse-sim` in Antioch's quickstart is an example directory name; this existing repository is your project directory.


## 2. Initialize Antioch in this repository root

The lean branch deliberately has no `pyproject.toml`. Run these **from `simulation/`**, not from its parent, `robots/`, or a new `warehouse-sim/` directory:

```bash
uv init --bare --python ">=3.12,<3.13"
uv python pin 3.12
uv add --compile-bytecode "antioch-sim[isaac-lab]>=0.5.0"
uv run antioch auth login
uv run antioch init
```

Use `isaac-lab` because the goal is an Isaac Lab learning project. That extra provides local editor types and selects the matching cloud starter; it does **not** install the full simulator on your computer. `antioch init` writes project files such as `antioch.yaml` and a starter script. Review those generated files before committing. See [Antioch's SDK quickstart](https://pypi.org/project/antioch-sim/0.5.338/).


## 3. Check the portable model locally

Still from the repository root:

```bash
ls robots/humanoid_legs_5/meshes/*.stl | wc -l
uv run --with mujoco --with numpy python robots/humanoid_legs_5/validate_mjcf.py
```

Expect 24 meshes, 10 actuators, 21 `qpos` entries, approximately 16 kg, and a maximum driven pin gap below 0.1 mm. This check runs MuJoCo locally and does not establish Isaac Sim compatibility.


## 4. Start the Antioch session and inspect its Isaac environment

```bash
uv run antioch session new
uv run antioch service exec -- python -c "import isaaclab, isaacsim; print(isaaclab.__file__); print(isaacsim.__file__)"
uv run antioch service exec -- bash -lc 'find / -path "*/scripts/tools/convert_mjcf.py" -print 2>/dev/null | head'
```

These commands run on Antioch's GPU service. Keep the selected SDK/engine versions in `pyproject.toml`, `uv.lock`, and `antioch.yaml` consistent. Check `uv run antioch service exec --help` if Antioch's CLI version changes. The local project files are synchronized to the service; generated USD should be saved as a project artifact rather than only in a temporary session filesystem.


## 5. Convert and inspect before training

Use the discovered `convert_mjcf.py` in Antioch's Isaac Lab environment to convert `robots/humanoid_legs_5/humanoid_legs_5.xml` to USD. The [Isaac Lab importer guide](https://isaac-sim.github.io/IsaacLab/main/source/how-to/import_new_asset.html) gives its current arguments. Run `--help` in the actual Antioch image first because Antioch may pin another version.

Use **PhysX** for the first physics check. Confirm that import preserves the two bar/crank pin constraints, the ten driven joints, four passive joints, mass/inertia, collision shapes, and feet contacting the ground. Step the imported robot and watch for separation, instability, or startup jumps. [Newton currently does not support closed chains](https://docs.isaacsim.omniverse.nvidia.com/6.2.0/physics/newton_physics.html).


## 6. Training gate

Before locomotion training, measure the real motor shaft axes, linkage zero offsets and range, motor-to-foot mapping, rotor inertia, and foot/bar inertias. The left Fusion upper-pin mismatch is 2.649 mm. The current MJCF directly actuates the foot and uses estimated motor placement and some estimated inertias. Existing 10-action policies cannot be loaded unchanged because the four passive joints change the state layout.

After PhysX and hardware checks pass, build an Isaac Lab articulation and velocity-command task. Select the ten controlled joints **by name**. Start with a small standing and commanded walking run, then scale on the RTX PRO 6000. Add SuperMimic or Puppeteer only after the base motion and reference data are sound.


## Evidence

- [Robot README](../robots/humanoid_legs_5/README.md)
- [MJCF](../robots/humanoid_legs_5/humanoid_legs_5.xml)
- [Validation report](../robots/humanoid_legs_5/humanoid_legs_5_validation.json)
- [Full robot sweep](../robots/humanoid_legs_5/humanoid_legs_5_full_sweep.mp4)
- [Ankle detail sweep](../robots/humanoid_legs_5/humanoid_legs_5_ankle_sweep.mp4)
